# Framework Comparison: Native SDK vs CrewAI vs LangGraph

**Docker image**: `ml4t`

The pipeline of notebooks 04 to 08 was built end to end with plain Python classes,
no orchestration framework. That choice keeps every transition visible and
every artifact inspectable, but it pushes the reader to write the
orchestration logic by hand. The agentic-framework ecosystem offers
alternatives that automate parts of that work in exchange for adopting a
framework's conventions.

This notebook expresses the same four-phase forecasting flow in three styles, and only two
of them are comparable:

1. **Native Python SDK**: direct composition of the chapter's
   `ResearchAgent` / `DebateAgent` / `SupervisorAgent` classes, same shape
   as `AIAForecaster` in [`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb).
2. **LangGraph**: the pipeline as a `StateGraph` with typed state and four
   nodes that delegate to the same specialist classes the native variant
   uses. Native and LangGraph call the same research, aggregation, debate and
   supervisor functions, so what differs between them is the orchestration and
   nothing about how evidence is accepted.
3. **CrewAI**: a **configuration sketch**, not a run. CrewAI's primitives
   (`Agent`, `Task`, `Crew`) are shown as they would be written, and the code is
   not executed here. An earlier pass reported fixed probabilities for this
   variant with no capture behind them; a number with nothing to inspect is worse
   than no number, so it is gone and the syntax is what remains.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 框架比較：原生 SDK、CrewAI 與 LangGraph

**Docker 映像檔**：`ml4t`

Notebook 04 至 08 的流程完全以一般 Python 類別建構，沒有使用協調框架。這讓每次階段轉換都清楚可見、每項產出都可檢視，但讀者必須自行撰寫協調邏輯。代理框架生態系提供其他選擇：採用框架的慣例，以換取部分工作的自動化。

本 Notebook 以三種風格表達相同的四階段預測流程，但其中只有兩種可以比較：

1. **原生 Python SDK**：直接組合本章的 `ResearchAgent`／`DebateAgent`／`SupervisorAgent` 類別，形式與 `AIAForecaster` 在 [`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb) 中的形式相同。
2. **LangGraph**：將流程表示為具有型別化狀態和四個節點的 `StateGraph`；節點委派給原生版本使用的相同專業類別。原生版本與 LangGraph 呼叫相同的研究、彙整、辯論與監督函式，因此兩者的差異只在協調方式，證據採納規則完全相同。
3. **CrewAI**：**設定草圖**，不是實際執行。此處展示 CrewAI 基本元件（`Agent`、`Task`、`Crew`）的寫法，但不執行程式碼。先前版本曾為此版本列出固定機率，卻沒有可供檢視的執行紀錄；無從查核的數字比沒有數字更糟，因此移除了數字，只保留語法。
<!-- ML4T_ZH_END c0000-p000 -->

So the comparison is of the **orchestration layer** between two implementations that share
their internals: how each expresses a four-phase flow, where state lives, and what happens
when something fails midway. It is a teaching implementation and not a performance
benchmark - two runs of one question decide nothing about which framework forecasts better.

**Learning objectives**:
- Express the same specialist, aggregate, debate and supervise flow twice, once by composing
  classes directly and once as a typed state graph, and read off what the graph adds
- Say where a run's state lives in each implementation, and find the entry point you would
  attach a debugger to when a phase fails midway
- Decide whether a project should adopt an orchestration framework, from what the framework
  takes over and what it costs to inspect a run through it

**Book reference**: §24.5 (The Engineering Stack: Frameworks and Migration).

**Prerequisites**: [`04_research_agent`](04_research_agent.ipynb) through
[`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb), which build the specialist
classes and the pipeline that the native and LangGraph variants compose.

**Replay vs live mode**: by default (`RUN_LIVE = False`) the notebook replays the pinned
2026-06-09 native and LangGraph captures, which hold the real model exchanges and the
durations measured that day. Those two ran under the chapter's earlier aggregation, so they
are labelled legacy where their numbers appear. No API call is made and the output is
identical on every machine. Set `RUN_LIVE = True` to run both variants against a current
question through the chapter's `LLMClient`, which costs money and reproduces nothing; use
`LLM_PROVIDER=mock` for the deterministic client.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
因此，這裡比較的是兩種內部元件相同的實作之**協調層**：各自如何表達四階段流程、狀態存放在哪裡，以及中途失敗時會發生什麼事。這是教學用實作，不是效能基準測試；針對一個問題各執行兩次，無法判斷哪個框架的預測較好。

**學習目標**：
- 將相同的專業研究、彙整、辯論與監督流程表達兩次：一次直接組合類別，一次使用型別化狀態圖，並辨識圖額外提供的功能
- 說明每種實作的執行狀態存放位置，並找出階段中途失敗時應連接除錯器的進入點
- 根據框架接手的工作，以及透過框架檢視執行過程所付出的代價，判斷專案是否應採用協調框架

**書籍參考**：§24.5（工程技術堆疊：框架與遷移）。

**先備知識**：[`04_research_agent`](04_research_agent.ipynb) 至 [`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb)；這些 Notebook 建立了原生版本與 LangGraph 版本共同組合的專業類別及流程。

**重播與即時模式**：預設（`RUN_LIVE = False`）重播固定的 2026-06-09 原生版本與 LangGraph 擷取紀錄，其中包含實際模型互動及當日測得的耗時。這兩次執行使用本章較早版本的彙整方法，因此出現數值時會標示為舊版。不會呼叫 API，且每台電腦的輸出都相同。將 `RUN_LIVE = True` 設定為即時執行，便會透過本章的 `LLMClient`，針對目前的問題執行兩種版本；這會產生成本，且結果無法重現。可使用 `LLM_PROVIDER=mock` 作為決定性的用戶端。
<!-- ML4T_ZH_END c0000-p001 -->


In [2]:
"""Framework comparison: one four-phase pipeline, native and as a LangGraph."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import math
import time
from datetime import date
from typing import TypedDict

import polars as pl
from agent_fixtures import get_chapter_contested_question
from agent_observability import TRACES_DIR, RunTrace, replay_llm_calls, trace_llm
from agent_pipeline import logodds_extremize, validate_probabilities
from agent_providers import TokenUsage, create_llm_client
from agent_research import ResearchAgent, format_agent_summary
from agent_schemas import AggregationResult, ForecastQuestion, ForecastResult
from agent_specialists import DebateAgent, SupervisorAgent
from agent_tools import create_search_client

## Settings

`RUN_LIVE` left at `False` replays the pinned 2026-06-09 run: the saved native and LangGraph
traces are reloaded, no API calls are made, and the reported figures are the ones measured
that day. Set it to `True` to run both executable variants live against a current question,
which costs money and reproduces nothing.

`N_AGENTS`, `MAX_STEPS`, `DEBATE_ROUNDS` and `SUPERVISOR_QUERIES` configure the chapter's
specialist classes, which the native and LangGraph variants both compose, so a change moves
both of them identically. That is what makes the statement count and the state-location
column the comparison, rather than the two probabilities.

The turn budget matters more here than elsewhere: an agent that runs out of turns returns no
probability at all, and a variant whose agents all did that would look like it had failed
when the budget was what failed.

`LLM_PROVIDER` is empty so the factory picks the first provider whose key is set. The CrewAI
section calls no model: it is printed configuration, and CrewAI's own LiteLLM layer is one
reason there was nothing for the chapter's tracing to capture.

<!-- ML4T_ZH_BEGIN c0002-p000 -->
## 設定

`RUN_LIVE` 保持為 `False` 時，會重播固定的 2026-06-09 執行：重新載入儲存的原生版本與 LangGraph 軌跡，不呼叫 API，並回報當日測得的數值。將其設為 `True`，則會針對目前的問題即時執行兩種可執行版本；這會產生成本，且結果無法重現。

`N_AGENTS`、`MAX_STEPS`、`DEBATE_ROUNDS` 與 `SUPERVISOR_QUERIES` 用於設定本章的專業類別；原生版本與 LangGraph 版本都組合這些類別，因此設定變更對兩者的影響相同。這正是比較應著眼於陳述式數量與狀態存放位置欄位，而非兩個機率的原因。

回合預算在此比其他地方更重要：代理若用盡回合，就完全不會回傳機率；若某版本的所有代理都如此，表面上會像是版本失敗，實際失敗的卻是預算設定。

`LLM_PROVIDER` 為空，讓工廠選擇第一個已設定金鑰的供應商。CrewAI 區段不呼叫模型，只印出設定；CrewAI 自己的 LiteLLM 層也是本章的追蹤機制無法擷取其呼叫的原因之一。
<!-- ML4T_ZH_END c0002-p000 -->


In [3]:
RUN_LIVE = False
LLM_PROVIDER = ""
N_AGENTS = 3
MAX_STEPS = 5
DEBATE_ROUNDS = 2
SUPERVISOR_QUERIES = 2
PINNED_TRACES = {
    "Native Python": "10_framework_comparison_20260609T150909Z_ba1855847f4c.json",
    "LangGraph": "10_framework_comparison_20260609T151202Z_c8cba3007562.json",
}

## Shared stages

The state retains all research attempts. Only supported probabilities enter the
mean. Abstention or a failed stage stops further model calls, preserving the
partial evidence. Confidence is self-reported, not a calibrated reliability score.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 共用階段

狀態保留所有研究嘗試。只有具證據支持的機率才會納入平均。棄權或階段失敗會停止後續模型呼叫，同時保留部分證據。信心程度由代理自行回報，並非經校準的可靠度分數。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
class ForecastState(TypedDict, total=False):
    question: ForecastQuestion
    artifacts: list
    aggregation: object
    debate: object
    supervisor: object
    final_probability: float | None
    status: str
    failure_reason: str


def research_node(state, llm, search):
    artifacts = [
        ResearchAgent(llm, search, agent_id=f"agent_{i}", max_steps=MAX_STEPS).run(
            state["question"], market_price=None
        )
        for i in range(N_AGENTS)
    ]
    return {"artifacts": artifacts}


def aggregate_node(state):
    probs = [a.p_yes for a in state["artifacts"] if a.status == "accepted" and a.p_yes is not None]
    if not probs:
        return {"status": "abstained", "failure_reason": "No supported research forecasts"}
    validate_probabilities(probs)
    return {
        "aggregation": AggregationResult("mean", sum(probs) / len(probs), input_probabilities=probs)
    }


def summaries(state):
    return "\n\n".join(
        format_agent_summary(a) for a in state["artifacts"] if a.status == "accepted"
    )


def debate_node(state, llm):
    agent = DebateAgent(llm, max_rounds=DEBATE_ROUNDS)
    try:
        debate = agent.run(
            state["question"].question, summaries(state), state["aggregation"].raw_probability
        )
    finally:
        state["debate"] = agent.artifact
    return {"debate": debate}


def supervise_node(state, llm, search):
    q = state["question"]
    agent = SupervisorAgent(llm, search, max_queries=SUPERVISOR_QUERIES)
    try:
        sup = agent.run(
            q.question,
            summaries(state),
            date.fromisoformat(q.cutoff_date) if q.cutoff_date else None,
        )
    finally:
        state["supervisor"] = agent.artifact
    probability = state["aggregation"].raw_probability
    debate = state["debate"]
    if debate.consensus_reached:
        probability = (debate.bull_final_probability + debate.bear_final_probability) / 2
    if sup.confidence == "high" and sup.p_yes is not None:
        probability = sup.p_yes
    return {"supervisor": sup, "final_probability": logodds_extremize(probability, math.sqrt(3))}


def apply_stage(state, function, *args):
    if state.get("status") != "accepted":
        return state
    try:
        return {**state, **function(state, *args)}
    except Exception as exc:
        return {**state, "status": "failed", "failure_reason": str(exc), "final_probability": None}


def forecast_result(state):
    total = TokenUsage()
    for artifact in [*state.get("artifacts", []), state.get("debate"), state.get("supervisor")]:
        if artifact is not None:
            total += artifact.token_usage
    return ForecastResult(
        question=state["question"],
        agents=state.get("artifacts", []),
        aggregation=state.get("aggregation"),
        debate=state.get("debate"),
        supervisor=state.get("supervisor"),
        final_probability=state.get("final_probability"),
        status=state["status"],
        failure_reason=state.get("failure_reason"),
        total_token_usage=total,
    )

## Native Python

Ordinary function calls own the state and execute the stages in order. Debugging
and failure handling remain visible in the Python call stack.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 原生 Python

一般函式呼叫持有狀態，並依序執行各階段。除錯與失敗處理都清楚呈現在 Python 呼叫堆疊中。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
def native_sdk_pipeline(question, llm, search):
    state = {"question": question, "status": "accepted"}
    for function, args in [
        (research_node, (llm, search)),
        (aggregate_node, ()),
        (debate_node, (llm,)),
        (supervise_node, (llm, search)),
    ]:
        state = apply_stage(state, function, *args)
    return forecast_result(state)

## LangGraph

The graph declares the same four stages and edges. Each node delegates to the
same function and retains the same artifacts. This example demonstrates the
execution graph; persistence/checkpointing requires separate configuration.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## LangGraph

圖宣告相同的四個階段與連線。每個節點都委派給相同函式，並保留相同的產出。此範例展示執行圖；持久化與檢查點功能需要另行設定。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
def langgraph_pipeline(question, llm, search):
    from langgraph.graph import END, StateGraph

    graph = StateGraph(ForecastState)
    graph.add_node("research", lambda s: apply_stage(s, research_node, llm, search))
    graph.add_node("aggregate", lambda s: apply_stage(s, aggregate_node))
    graph.add_node("debate", lambda s: apply_stage(s, debate_node, llm))
    graph.add_node("supervise", lambda s: apply_stage(s, supervise_node, llm, search))
    graph.set_entry_point("research")
    graph.add_edge("research", "aggregate")
    graph.add_edge("aggregate", "debate")
    graph.add_edge("debate", "supervise")
    graph.add_edge("supervise", END)
    state = graph.compile().invoke({"question": question, "status": "accepted"})
    return forecast_result(state)

## Exercise both orchestrators with a failure fixture

This explicitly synthetic client attempts an unsupported forecast. Neither
orchestrator should produce an aggregate or final probability. No timing or
accuracy claims are made from this test.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 以失敗測試案例演練兩種協調器

這個明確標示為合成資料的用戶端會嘗試產生缺乏證據支持的預測。兩種協調器都不應產生彙整結果或最終機率。本測試不據此宣稱任何耗時或準確度結果。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
class UnsupportedForecastFixture:
    model_name = "synthetic-unsupported-forecast"

    def complete_with_usage(self, messages, json_mode=True):
        return '{"action":"forecast","p_yes":0.9,"rationale":"No evidence"}', TokenUsage()


for pipeline in (native_sdk_pipeline, langgraph_pipeline):
    result = pipeline(
        ForecastQuestion("Synthetic unsupported question?"), UnsupportedForecastFixture(), None
    )
    assert result.status == "abstained" and result.final_probability is None
    assert result.aggregation is None
    print(f"{pipeline.__name__}: {result.status}; {len(result.agents)} attempts retained")

native_sdk_pipeline: abstained; 3 attempts retained


langgraph_pipeline: abstained; 3 attempts retained


## CrewAI syntax example

CrewAI uses agents, tasks and a process. The following configuration is shown as
source, not executed. It has no research tools or trace integration and therefore
cannot be compared as an evidence-backed forecasting pipeline. Implement those
contracts before using its output as a researched forecast.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## CrewAI 語法範例

CrewAI 使用代理、任務與流程。以下設定僅以原始碼形式展示，並未執行。它沒有研究工具或軌跡整合，因此無法作為有證據支持的預測流程來比較。使用其輸出作為經研究的預測前，必須先實作這些契約。
<!-- ML4T_ZH_END c0012-p000 -->


In [8]:
CREWAI_EXAMPLE = """from crewai import Agent, Task, Crew, Process
analyst = Agent(role="Research analyst", goal="Review supplied evidence", backstory="Forecasting analyst", llm=model)
task = Task(description=question_with_evidence, expected_output="Probability and source-linked rationale", agent=analyst)
crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
# A live application would attach tools, validate outputs and retain actual messages.
"""
print(CREWAI_EXAMPLE)
print("CrewAI: syntax only; no probability, timing or performance measurement")

from crewai import Agent, Task, Crew, Process
analyst = Agent(role="Research analyst", goal="Review supplied evidence", backstory="Forecasting analyst", llm=model)
task = Task(description=question_with_evidence, expected_output="Probability and source-linked rationale", agent=analyst)
crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
# A live application would attach tools, validate outputs and retain actual messages.

CrewAI: syntax only; no probability, timing or performance measurement


## Inspect measured captures or run live

The default table reports the original measured durations and probabilities
without pretending to rerun today's implementation. A fresh live run retains full
research, stage outputs, actual messages and timing. Different stochastic model
calls are not a controlled test of framework effects.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 檢視實測擷取紀錄或即時執行

預設表格回報原始測得的耗時與機率，不會假裝重新執行了目前的實作。新的即時執行會保留完整研究、各階段輸出、實際訊息與耗時。不同的隨機模型呼叫並非控制框架效應的實驗。
<!-- ML4T_ZH_END c0014-p000 -->


In [9]:
rows, records = [], []
if RUN_LIVE:
    llm = create_llm_client(LLM_PROVIDER)
    search = create_search_client()
    question = get_chapter_contested_question()
    for label, pipeline in [
        ("Native Python", native_sdk_pipeline),
        ("LangGraph", langgraph_pipeline),
    ]:
        tracer = trace_llm(llm, label=label)
        started = time.perf_counter()
        result = pipeline(question, tracer, search)
        result.duration_seconds = time.perf_counter() - started
        run = RunTrace.from_result(
            result,
            notebook="10_framework_comparison",
            provider=llm.model_name,
            params={
                "variant": label,
                "n_agents": N_AGENTS,
                "max_steps": MAX_STEPS,
                "debate_rounds": DEBATE_ROUNDS,
                "supervisor_queries": SUPERVISOR_QUERIES,
                "statistical_correction_a": math.sqrt(3),
            },
            llm_calls=tracer.calls,
        )
        print(f"Saved {run.save().name}")
        records.append(run)
        rows.append(
            {
                "variant": label,
                "mode": run.execution_mode,
                "status": result.status,
                "duration_s": result.duration_seconds,
                "final_p": result.final_probability,
            }
        )
else:
    for label, filename in PINNED_TRACES.items():
        run = RunTrace.load(TRACES_DIR / filename)
        records.append(run)
        rows.append(
            {
                "variant": label,
                "mode": "replay",
                "status": "legacy_capture",
                "duration_s": run.params["elapsed_s"],
                "final_p": run.params["result"]["final_p"],
            }
        )
        print(f"{label}: {run.created_at}; {len(run.llm_calls)} actual captured model calls")
print(pl.DataFrame(rows))
print("CrewAI omitted: no inspectable measured capture")

Native Python: 2026-06-09T15:09:09+00:00; 21 actual captured model calls
LangGraph: 2026-06-09T15:12:02+00:00; 19 actual captured model calls
shape: (2, 5)
┌───────────────┬────────┬────────────────┬────────────┬─────────┐
│ variant       ┆ mode   ┆ status         ┆ duration_s ┆ final_p │
│ ---           ┆ ---    ┆ ---            ┆ ---        ┆ ---     │
│ str           ┆ str    ┆ str            ┆ f64        ┆ f64     │
╞═══════════════╪════════╪════════════════╪════════════╪═════════╡
│ Native Python ┆ replay ┆ legacy_capture ┆ 108.0      ┆ 0.3     │
│ LangGraph     ┆ replay ┆ legacy_capture ┆ 102.7      ┆ 0.48    │
└───────────────┴────────┴────────────────┴────────────┴─────────┘
CrewAI omitted: no inspectable measured capture


In [10]:
print(replay_llm_calls(records[0].call_log()[:2], content_chars=600))
print("Full messages remain in the saved JSON; display excerpts are abbreviated.")

  CALL #0 [native]  ·  claude-sonnet-4-20250514  ·  json_mode=True  ·  tokens in/out=254/26
------------------------------------------------------------------------------
  ▸ SYSTEM:
      You are a forecasting agent in a multi-agent forecasting system.
      Your job:
      1) Gather evidence by issuing web/news search queries when needed.
      2) Then produce a binary probability forecast for the question.
      You must follow the action schema exactly and output valid JSON only.
      You must not browse prediction market prices unless they are explicitly
      provided.
  ▸ USER:
      QUESTION:
      Will the Federal Reserve hike rates in 2026?
      MARKET CONTEXT:
      Resolves YES if the upper bound of the federal funds target rate is
      increased at any point between January 1, 2026 and the Fed's December
      2026 meeting (scheduled December 8-9, 2026).
      MARKET IMPLIED PROBABILITY (p_yes):
      0.545
      NEXT ACTION SCHEMA (output JSON only):
      If you need 

The statement count covers the orchestration layer, not the agent classes.
CrewAI is the most verbose at the orchestration layer because each agent
carries a role / goal / backstory triple and each task carries a
description / expected_output / context list. LangGraph's overhead is
the typed state schema and four short node functions; once the schema is
defined, each node is a handful of statements. Native SDK is a single
function.

The two `final_p` values are not expected to match, and their gap measures nothing about
orchestration: native and LangGraph run the same specialist classes over the same evidence
rule, so what separates their numbers is that they are two samples from a stochastic model.
That is the argument for comparing structure - statements written, where state lives, where a
debugger attaches - rather than forecast agreement.

## Comparison matrix


<!-- ML4T_ZH_BEGIN c0017-p000 -->
陳述式數量只涵蓋協調層，不含代理類別。CrewAI 在協調層最冗長，因為每個代理都帶有角色／目標／背景故事三項設定，每個任務也帶有描述／預期輸出／脈絡清單。LangGraph 的額外負擔是型別化狀態結構與四個簡短的節點函式；結構定義好後，每個節點只需少量陳述式。原生 SDK 則是一個函式。

兩個 `final_p` 值不必相符，而其差距也無法衡量協調方式：原生版本與 LangGraph 版本在相同證據規則下使用相同的專業類別，因此數值差異來自隨機模型的兩次抽樣。這正是應比較結構（撰寫的陳述式數量、狀態存放位置、除錯器的連接點），而非預測是否一致的理由。

## 比較矩陣
<!-- ML4T_ZH_END c0017-p000 -->

| Dimension | Native SDK | CrewAI | LangGraph |
|---|---|---|---|
| **Orchestration shape** | function call stack | role-based agents in a `Crew` | `StateGraph` with typed nodes |
| **State location** | local variables | task outputs threaded by framework | explicit `TypedDict` |
| **Parallelism** | manual (threads / asyncio) | opt-in asynchronous tasks | conditional / parallel edges |
| **Checkpointing** | write it yourself, as in notebook 03 | not first-class | first-class (`BaseCheckpointSaver`) |
| **LLM layer** | chapter `LLMClient` | LiteLLM (provider auto-detect) | passes through whatever the nodes use |
| **Debug entry point** | the line that raised | inside CrewAI's loop | `app.get_state(thread)` snapshot |
| **Full-trace capture** | one client wrapper | framework callbacks / verbose logs | one client wrapper |
| **Visual debugging** | print / pdb | verbose logs | graph rendering + state inspection |
| **Dep weight** | LLM SDK only | crewai + langchain transitives | langgraph + langchain transitives |
| **Learning curve** | low (Python only) | medium (roles / tasks / processes) | medium-high (graph + state) |

The differences that matter for a Chapter 24-shaped pipeline cluster in
two columns: LangGraph exposes an explicit checkpoint abstraction, while
native Python adds no orchestration dependency. Migration paths, observability stories, framework-momentum
trade-offs, and a longer discussion of when persistence /
parallelism / conditional-flow pressures justify a framework live in
§24.5 of the book rather than in the notebook.

## Key Takeaways


<!-- ML4T_ZH_BEGIN c0017-p001 -->
| 面向 | 原生 SDK | CrewAI | LangGraph |
|---|---|---|---|
| **協調形式** | 函式呼叫堆疊 | `Crew` 中依角色設定的代理 | 具有型別化節點的 `StateGraph` |
| **狀態存放位置** | 區域變數 | 由框架串接的任務輸出 | 明確的 `TypedDict` |
| **平行處理** | 手動（執行緒／asyncio） | 可選擇啟用非同步任務 | 條件式／平行連線 |
| **檢查點** | 自行撰寫，如 Notebook 03 | 非第一級功能 | 第一級功能（`BaseCheckpointSaver`） |
| **LLM 層** | 本章的 `LLMClient` | LiteLLM（自動偵測供應商） | 沿用節點所使用的用戶端 |
| **除錯進入點** | 引發例外的程式碼行 | CrewAI 迴圈內部 | `app.get_state(thread)` 快照 |
| **完整軌跡擷取** | 一層用戶端包裝 | 框架回呼／詳細日誌 | 一層用戶端包裝 |
| **視覺化除錯** | print／pdb | 詳細日誌 | 圖形呈現與狀態檢視 |
| **相依套件負擔** | 只有 LLM SDK | crewai 與間接相依的 langchain 套件 | langgraph 與間接相依的 langchain 套件 |
| **學習曲線** | 低（只有 Python） | 中（角色／任務／流程） | 中高（圖與狀態） |

對第 24 章這類流程而言，重要差異集中在兩個面向：LangGraph 提供明確的檢查點抽象層，而原生 Python 不增加協調套件相依性。遷移路徑、可觀測性、框架發展動能的取捨，以及何時持久化／平行處理／條件式流程的需求足以支持採用框架，詳見書籍 §24.5，而非本 Notebook。

## 重點整理
<!-- ML4T_ZH_END c0017-p001 -->

1. **Start with plain Python for a pipeline this shape.** It asks the caller to write the
   least orchestration and puts every failure on the line that caused it. That is structural
   evidence about this pipeline, not a ranking of the frameworks.
2. **A framework earns its place against a named pressure, not a preference.** Crash recovery
   mid-run is what LangGraph's checkpoint abstraction is for; named personas with distinct
   charters are what CrewAI's role and goal slots are for. Without one of those pressures, the
   dependency and the indirection are the whole trade.
3. **Ask where the run's state lives before adopting anything.** It decides what you can
   inspect when a run goes wrong: local variables you can print, a framework's task outputs,
   or a typed state object you can snapshot.
4. **A framework that owns the model client owns the observability.** CrewAI drives LiteLLM
   directly, so nothing in this chapter's tracing sees its calls. That is why it appears here
   as syntax rather than as a run: there was no capture to inspect, and the concrete cost of
   the boundary moving is that there could not be one.
5. **Two probabilities from two runs are not a benchmark.** The native and LangGraph variants
   share their internals, so what differs between their numbers is sampling, not
   orchestration. Comparing frameworks needs a structural measure, which is what the
   statement count and the state-location column are for.


<!-- ML4T_ZH_BEGIN c0017-p002 -->
1. **這類流程先從一般 Python 開始。** 呼叫端需要撰寫的協調邏輯最少，且每次失敗都會指向造成失敗的程式碼行。這是針對此流程的結構性證據，不是框架排名。
2. **框架是否值得採用，取決於明確需求，而非偏好。** LangGraph 的檢查點抽象層用於執行中途當機後的復原；CrewAI 的角色與目標欄位適合有不同職責的具名角色。若沒有這些需求，增加的相依性與間接性就是全部代價。
3. **採用任何工具前，先確認執行狀態存放在哪裡。** 這決定執行出錯時能檢視什麼：可印出的區域變數、框架的任務輸出，或可擷取快照的型別化狀態物件。
4. **掌控模型用戶端的框架，也掌控可觀測性。** CrewAI 直接驅動 LiteLLM，因此本章的追蹤機制看不到它的呼叫。這就是此處僅展示語法而非執行結果的原因：沒有可供檢視的擷取紀錄，而邊界轉移的具體代價就是無法取得該紀錄。
5. **兩次執行產生的兩個機率並非基準測試。** 原生版本與 LangGraph 版本共用內部元件，所以兩者數值的差異來自抽樣，而非協調方式。比較框架需要結構性指標，陳述式數量與狀態存放位置欄位正是為此而設。
<!-- ML4T_ZH_END c0017-p002 -->

**Known limitations of what is built here.** One question, one run per variant, one model, and
only two of the three variants execute at all. The statement count is a proxy for how much a
caller writes and says nothing about how much there is to understand: a framework can trade
statements for concepts. The elapsed times are single measurements of two different call paths
on one machine, taken under the chapter's earlier aggregation, and the comparison matrix
describes framework versions that move.

**Next**: [`11_research_operator`](11_research_operator.ipynb) replays a
production-shaped operator loop against a real case-study registry.

**Book**: Section 24.5 covers migration paths, the persistence, parallelism and
conditional-flow pressures that justify a framework, and the ecosystem-version note.

<!-- ML4T_ZH_BEGIN c0017-p003 -->
**此處實作的已知限制。** 只有一個問題、每種版本各執行一次、使用一個模型，而且三種版本中只有兩種真正執行。陳述式數量只是呼叫端程式碼量的替代指標，無法反映理解程式的難度：框架可能以較少陳述式換取更多概念。耗時是在同一台電腦上，針對兩條不同呼叫路徑各測得的一次數值，且使用本章較早的彙整方法；比較矩陣描述的框架版本也會變動。

**下一步**：[`11_research_operator`](11_research_operator.ipynb) 會針對真實案例研究登錄資料，重播接近正式環境的操作代理迴圈。

**書籍**：第 24.5 節討論遷移路徑、持久化／平行處理／條件式流程等支持採用框架的需求，以及生態系版本的說明。
<!-- ML4T_ZH_END c0017-p003 -->
